# Uber (ride-hailing)

> **Time-box:** 45–60 minutes. The trip state machine is the heart of the design.

## Core requirements

1. Users sign up as a **rider** or a **driver**. (One person may eventually do both — your call on how to model.)
2. Drivers report their location periodically.
3. A rider requests a ride from pickup → dropoff. The system finds a nearby driver.
4. The ride moves through a **state machine**: `requested → accepted → en_route_to_pickup → in_progress → completed`. Also `cancelled` (by rider or driver, from certain states only).
5. After completion, rider and driver can rate each other (1–5 stars).
6. A user can list their past rides.

## Stretch goals

- Surge pricing — a multiplier table by area/time?
- Multiple ride classes (UberX, Black, …) and per-class pricing.
- Driver earnings rollup.
- Payments (intentionally hand-wave — design the boundary, don't build it).

## Things the interviewer will probe

- **Geo lookup:** how do you find "nearby drivers"? In production: H3 / geohash / PostGIS. In this exercise: a naive bounding-box filter on lat/lng is fine — but **mention** what you'd do at scale.
- **State machine integrity:** how do you stop a driver `accepting` a ride that was already `cancelled`? Where does the state transition live — in the API, in a DB CHECK, or in a stored procedure?
- **Driver locations:** one row per driver (overwrite) vs append-only stream? Trade-offs?
- **Concurrent acceptance:** two drivers accept the same request. How do you guarantee only one wins?
- **Soft vs hard delete** for cancelled rides.
- **Modelling users:** one `users` table with a `type` column? Separate `riders` and `drivers`? A single users table plus role tables? Defend a choice.

---
## Setup

In [ ]:
import json
import sqlite3

import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from IPython.display import display
from pydantic import BaseModel

## Schema

Edit the SQL and re-run this cell to get a fresh in-memory database.

In [ ]:
SCHEMA = """
-- Design your tables here
"""

conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(SCHEMA)

print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()])

## API

> After editing any cell below, re-run from **App** down through **Client**.

In [ ]:
# ── App + models ──────────────────────────────────────────────────────────────
app = FastAPI(title="Uber")

In [ ]:
# ── Endpoints ─────────────────────────────────────────────────────────────────
@app.get("/healthz")
def healthz():
    return {"status": "ok"}

In [ ]:
# ── Client ────────────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=True)
print(client.get("/healthz").json())

## Helpers

In [ ]:
def call(method: str, path: str, **kwargs):
    r = getattr(client, method)(path, **kwargs)
    body = r.json() if r.content else None
    print(f"{method.upper():6s} {path}  →  {r.status_code}")
    if body is not None:
        print(json.dumps(body, indent=2))
    return r


def df(table: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM {table}", conn)


def query(sql: str, *params) -> pd.DataFrame:
    return pd.read_sql(sql, conn, params=list(params) if params else None)


def show_all():
    names = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()]
    for name in names:
        count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"\n── {name} ({count} rows) ──")
        display(pd.read_sql(f"SELECT * FROM {name}", conn))

## Demo

In [ ]:
# Call your endpoints here

## All tables

In [ ]:
show_all()